# Track B — H4 재검증 (track_b_08)

담당: 김태헌 | 선행 노트북: track_b_05(H4 최초 확인), track_b_07(튜닝 방법론)

## 이 노트북의 목적

track_b_05의 H4 결과(기존 신용정보 26개 AUC 0.8076 → 기존+대안변수 95개 AUC 0.8382, +3.07%p)는 **고정 하이퍼파라미터 + 1회 분할** 기준 1차 확인치였다. track_b_07에서 세 알고리즘(로지스틱/RF/XGBoost)을 튜닝 비교한 결과 **XGBoost가 PR-AUC 기준 최고 성능**으로 확정됐으므로, 이 노트북은 XGBoost 하나로 다음 두 모델을 각각 튜닝+5-fold CV하여 H4를 정식 재확정한다.

- **Model A (기존 신용정보만)**: `track_b_traditional_credit_features_for_H4comparison.csv` 26개 핵심변수(+ 결측플래그 8개, 총 34개 컬럼)
- **Model B (기존+대안변수)**: Model A + `track_b_features_train_v2.csv`의 69개 대안변수(원-핫 인코딩 포함 90개) = 총 124개

두 파일은 CUST_ID·TARGET이 완전히 동일한 신용이력 보유군 285,890명이라 CUST_ID 기준으로 그대로 병합 가능하다(사전 확인 완료).


## 0. 환경 설정

In [1]:
# 로컬 환경: 터미널에서 아래 명령어로 한 번만 설치해두면 됨
# pip install "xgboost<3.0" shap

import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score
import xgboost as xgb
import shap

RANDOM_STATE = 42


## 1. 데이터 로드 및 병합

`handoff_path`는 track_b_07과 동일한 폴더를 사용한다.

In [2]:
handoff_path = r'C:\Users\tehun\Desktop\multicamp\프로젝트\creditscore\cardCB'  # 필요시 수정

df_h4 = pd.read_csv(f'{handoff_path}/track_b_traditional_credit_features_for_H4comparison.csv')
df_train = pd.read_csv(f'{handoff_path}/track_b_features_train_v2.csv')

print(f"H4 비교용(26개+결측플래그): {df_h4.shape}")
print(f"69개 변수(train_v2): {df_train.shape}")

# 동일 모집단 재확인
assert set(df_h4['CUST_ID']) == set(df_train['CUST_ID']), "CUST_ID 불일치!"
assert (df_h4.set_index('CUST_ID')['TARGET'] == df_train.set_index('CUST_ID')['TARGET']).all(), "TARGET 불일치!"
print("모집단 및 TARGET 일치 확인 완료")


H4 비교용(26개+결측플래그): (285890, 36)
69개 변수(train_v2): (285890, 71)
모집단 및 TARGET 일치 확인 완료


## 2. Model A / Model B 변수셋 구성

- Model A: `df_h4`의 CUST_ID·TARGET을 제외한 전체 컬럼(26개 핵심 + 결측플래그 8개 = 34개)
- Model B: Model A + `df_train`의 69개 변수(범주형 `JB_TP`/`HOME_ADM`은 원-핫 인코딩)

In [3]:
traditional_cols = [c for c in df_h4.columns if c not in ['CUST_ID', 'TARGET']]
alt_cols = [c for c in df_train.columns if c not in ['CUST_ID', 'TARGET']]
categorical_cols = ['JB_TP', 'HOME_ADM']

# CUST_ID 기준 병합 (TARGET은 한쪽만 사용)
df_merged = df_h4.merge(df_train.drop(columns=['TARGET']), on='CUST_ID', how='inner')
print(f"병합 결과: {df_merged.shape}")

y = df_merged['TARGET']

# Model A: 기존 신용정보 26개(+결측플래그)만
X_A = df_merged[traditional_cols]

# Model B: 기존 + 대안변수(69개, 원-핫 포함)
X_B_raw = df_merged[traditional_cols + alt_cols]
X_B = pd.get_dummies(X_B_raw, columns=categorical_cols, prefix=categorical_cols)

print(f"Model A 변수 개수: {X_A.shape[1]}개")
print(f"Model B 변수 개수: {X_B.shape[1]}개 (원-핫 인코딩 후)")


병합 결과: (285890, 105)
Model A 변수 개수: 34개
Model B 변수 개수: 124개 (원-핫 인코딩 후)


## 3. Train / Test 분리

Model A와 Model B가 **완전히 같은 고객**을 같은 방식으로 나누도록, 인덱스 기준으로 한 번만 분리해서 A/B에 동일하게 적용한다.

In [4]:
idx_train, idx_test = train_test_split(
    df_merged.index, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

X_A_train, X_A_test = X_A.loc[idx_train], X_A.loc[idx_test]
X_B_train, X_B_test = X_B.loc[idx_train], X_B.loc[idx_test]
y_train, y_test = y.loc[idx_train], y.loc[idx_test]

print(f"Train: {len(idx_train)} / Test: {len(idx_test)}")
print(f"Train 양성비율: {y_train.mean():.4%} / Test 양성비율: {y_test.mean():.4%}")


Train: 228712 / Test: 57178
Train 양성비율: 4.2280% / Test 양성비율: 4.2289%


## 4. 교차검증 설계 (track_b_07과 동일)

In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
PRIMARY_SCORING = 'average_precision'

xgb_param_dist = {
    'n_estimators': [200, 300, 500, 800],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 5, 10],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 5, 10],
}

def tune_xgb(X_tr, y_tr, label):
    spw = (y_tr == 0).sum() / (y_tr == 1).sum()
    search = RandomizedSearchCV(
        xgb.XGBClassifier(
            scale_pos_weight=spw, eval_metric='aucpr',
            random_state=RANDOM_STATE, n_jobs=-1, tree_method='hist',
        ),
        xgb_param_dist, n_iter=30, scoring=PRIMARY_SCORING,
        cv=cv, random_state=RANDOM_STATE, n_jobs=-1, verbose=1
    )
    search.fit(X_tr, y_tr)
    print(f"[{label}] 최적 파라미터: {search.best_params_}")
    print(f"[{label}] CV PR-AUC: {search.best_score_:.4f}")
    return search


## 5. Model A 튜닝 (기존 신용정보 26개)

In [6]:
search_A = tune_xgb(X_A_train, y_train, 'Model A (26개)')


Fitting 5 folds for each of 30 candidates, totalling 150 fits
[Model A (26개)] 최적 파라미터: {'subsample': 0.6, 'reg_lambda': 10, 'reg_alpha': 0, 'n_estimators': 800, 'min_child_weight': 5, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 1.0}
[Model A (26개)] CV PR-AUC: 0.2781


## 6. Model B 튜닝 (기존+대안변수 124개)

In [7]:
search_B = tune_xgb(X_B_train, y_train, 'Model B (26+69개)')


Fitting 5 folds for each of 30 candidates, totalling 150 fits
[Model B (26+69개)] 최적 파라미터: {'subsample': 1.0, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 800, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 0.6}
[Model B (26+69개)] CV PR-AUC: 0.4355


## 7. H4 최종 비교

track_b_05 원본 표와 같은 형식(AUC, PR-AUC, 개선폭)으로 정리.

In [8]:
def eval_test(search, X_te, y_te):
    proba = search.best_estimator_.predict_proba(X_te)[:, 1]
    return roc_auc_score(y_te, proba), average_precision_score(y_te, proba)

auc_A, prauc_A = eval_test(search_A, X_A_test, y_test)
auc_B, prauc_B = eval_test(search_B, X_B_test, y_test)

h4_result = pd.DataFrame([
    {'model': '기존 신용정보만(26개)', 'test_auc': auc_A, 'test_prauc': prauc_A},
    {'model': '기존+대안변수(26+69개)', 'test_auc': auc_B, 'test_prauc': prauc_B},
])
h4_result['auc_개선폭(%p)'] = [None, (auc_B - auc_A) * 100]
h4_result['prauc_개선폭(%p)'] = [None, (prauc_B - prauc_A) * 100]

h4_result


,model,test_auc,test_prauc,auc_개선폭(%p),prauc_개선폭(%p)
0,기존 신용정보만(26개),0.822154,0.275637,NaN,NaN
1,기존+대안변수(26+69개),0.908534,0.435314,8.637984,15.967616


## 8. 결과 저장

track_b_05의 H4 방향성 확인치(+3.07%p / +3.64%p)와 이번 정식 재확정치를 함께 기록해둔다.

In [9]:
h4_result.to_csv(f'{handoff_path}/track_b_h4_revalidation_final.csv', index=False, encoding='utf-8-sig')

best_params_h4 = {
    'model_A_26vars': search_A.best_params_,
    'model_B_26plus69vars': search_B.best_params_,
    'track_b_05_1st_pass_auc_gain_pp': 3.07,   # 참고: 서윤 1차 확인치
    'track_b_05_1st_pass_prauc_gain_pp': 3.64,  # 참고: 서윤 1차 확인치
    'final_auc_gain_pp': float((auc_B - auc_A) * 100),
    'final_prauc_gain_pp': float((prauc_B - prauc_A) * 100),
}

with open(f'{handoff_path}/track_b_h4_best_params.json', 'w', encoding='utf-8') as f:
    json.dump(best_params_h4, f, ensure_ascii=False, indent=2, default=str)

print("저장 완료: track_b_h4_revalidation_final.csv, track_b_h4_best_params.json")
print(h4_result)


저장 완료: track_b_h4_revalidation_final.csv, track_b_h4_best_params.json
             model  test_auc  test_prauc  auc_개선폭(%p)  prauc_개선폭(%p)
0    기존 신용정보만(26개)  0.822154    0.275637          NaN            NaN
1  기존+대안변수(26+69개)  0.908534    0.435314     8.637984      15.967616


## 다음 단계 (참고)

- **H7 재검증**(선택): Model B의 SHAP 순위에서 기존 신용정보 26개 변수들의 순위가 Model A 대비 얼마나 유지되는지 확인하면 track_b_05의 "보완 관계" 결론도 같은 방식으로 재확정 가능. 필요하면 별도로 진행.
